# How to generate a metareport?

### Create a metareport comparing synthetic datasets with respect to a list of metrics. /!\ Only for the summary.

Assume that the synthetic datasets to compare are already generated \
Based on the Adult Dataset

In [1]:
# Standard library
import sys
import tempfile
from pathlib import Path

sys.path.append("..")

# Local packages
import config

# 3rd party packages
import pandas as pd
import utils.draw
from metrics.metareport import Metareport

## Load the real and synthetic Adult datasets

In [2]:
df_real = {}
df_real["train"] = pd.read_csv("../data/adult_train.csv")
df_real["test"] = pd.read_csv("../data/adult_test.csv")
df_real["train"].shape

(26048, 15)

### Choose the synthetic dataset

In [3]:
# generated by Synthpop here
df_synth = {
    "train": pd.read_csv("../results/data/2024-01-04_Synthpop_26048samples.csv"),
    "test": pd.read_csv("../results/data/2024-01-04_Synthpop_6513samples.csv"),
    "2nd_gen": pd.read_csv(
        "../results/data/2024-01-04_Synthpop_26048samples_2nd_gen.csv"
    ),
}

# random synthetic dataset to compare to the one generated by Synthpop
df_mock = {
    "train": df_real["train"].apply(
        lambda x: x.sample(frac=1, replace=True).to_numpy()
    ),
    "test": df_real["test"].apply(lambda x: x.sample(frac=1, replace=True).to_numpy()),
    "2nd_gen": df_synth["train"].apply(
        lambda x: x.sample(frac=1, replace=True).to_numpy()
    ),
}

synth_datasets = {"synthpop": df_synth, "random": df_mock}

## Configure the metadata dictionary

### The continuous and categorical variables need to be specified, as well as the variable to predict for the future learning task

In [5]:
metadata = {
    "continuous": [
        "age",
        "fnlwgt",
        "education-num",
        "capital-gain",
        "capital-loss",
        "hours-per-week",
    ],
    "categorical": [
        "workclass",
        "education",
        "marital-status",
        "occupation",
        "relationship",
        "race",
        "sex",
        "native-country",
        "salary",
    ],
    "variable_to_predict": "salary",
}

## Generate the metareport

In [6]:
parameters = {  # see the notebooks utility_report and privacy_report for more details
    "cross_learning": True,
    "num_repeat": 1,
    "num_kfolds": 3,
    "num_optuna_trials": 15,
    "use_gpu": True,
    "sampling_frac": 1,
}

In [7]:
metareport = Metareport(
    dataset_name="Adult Dataset",
    df_real=df_real,
    synthetic_datasets=synth_datasets,
    metadata=metadata,
    figsize=(8, 6),  # will be automatically adjusted for larger or longer figures
    random_state=0,  # for reproducibility purposes
    metareport_folderpath=None,  # a dictionary containing the path of each already computed report to load and compare
    metrics=None,  # list of the metrics to compute. Can be utility or privacy metrics. If not specified, all the metrics are computed.
    params=parameters,  # the dictionary containing the parameters for both utility and privacy reports
)

In [8]:
metareport.compute()

## Get the summary report as a pandas dataframe

In [9]:
df_summary = metareport.summary()

In [11]:
df_summary

compared,random,synthpop
metric,,
cat_consis-within_ratio,1.0,1.0
cat_stats-frequency_coverage,0.999087,0.998076
cat_stats-support_coverage,1.0,1.0
classif-diff_real_synth,0.452507,0.014791
collision-avg_num_appearance_collision_real,[-],1.012987
collision-avg_num_appearance_collision_synth,[-],1.064935
collision-avg_num_appearance_realcontrol,1.000154,1.000154
collision-avg_num_appearance_realtrain,1.000615,1.000615
collision-avg_num_appearance_synth,1.0,1.010278


### Style the result

The best value (minimum or maximal according to the submetric objective) is colored in green. The worst in yellow.

In [12]:
# s = df_summary.style.pipe(Metareport.make_pretty, metrics=list(df_summary.index))
# s

### Save the styled result as html

In [10]:
with tempfile.TemporaryDirectory() as temp_dir:
    with open(Path(temp_dir) / "df.html", "w") as f:
        print(s.to_html(), file=f)

## Save and load the metareport

In [11]:
with tempfile.TemporaryDirectory() as temp_dir:
    metareport.save(savepath=temp_dir)  # save
    new_report = Metareport(
        metareport_folderpath={"synthpop": temp_dir, "random": temp_dir}
    )  # load